In [5]:
# ======================================================
# مشروع: Data Pipeline Scalability Lab
# إعداد المبرمجة: منار سلمان (Manar Salman)
# ======================================================

import warnings
warnings.filterwarnings('ignore')

# 1.  المكتبات
!pip install -q cudf-cu12 dask[complete] gradio plotly

import pandas as pd
import dask.dataframe as dd
import cudf
import numpy as np
import time
import os
import gradio as gr
import plotly.graph_objects as go

#  الملفات
DATA_PATH = 'large_dataset.csv'

# ---   (Backend Logic) ---

def generate_data(num_rows):
    """توليد بيانات عشوائية ضخمة لمحاكاة الواقع"""
    df = pd.DataFrame({
        'id': range(num_rows),
        'category': np.random.choice(['Electronics', 'Clothing', 'Home', 'Toys', 'Books'], num_rows),
        'quantity': np.random.randint(1, 20, size=num_rows),
        'price': np.random.uniform(5.0, 1000.0, size=num_rows),
        'date': pd.date_range(start='2024-01-01', periods=num_rows, freq='S')
    })
    df.to_csv(DATA_PATH, index=False)
    return DATA_PATH

def run_benchmark(filepath):
    """مقارنة الأداء بين محركات المعالجة المختلفة"""
    results = {}

    # by Pandas (CPU)
    start = time.time()
    df_pd = pd.read_csv(filepath)
    df_pd['total'] = df_pd['quantity'] * df_pd['price']
    _ = df_pd.groupby('category')['total'].sum()
    results['Pandas (Standard CPU)'] = time.time() - start

    # by Dask (Parallel CPU)
    start = time.time()
    ddf = dd.read_csv(filepath)
    ddf['total'] = ddf['quantity'] * ddf['price']
    _ = ddf.groupby('category')['total'].sum().compute()
    results['Dask (Parallel CPU)'] = time.time() - start

    # by cuDF (GPU)
    try:
        start = time.time()
        gdf = cudf.read_csv(filepath)
        gdf['total'] = gdf['quantity'] * gdf['price']
        _ = gdf.groupby('category')['total'].sum()
        results['cuDF (NVIDIA GPU)'] = time.time() - start
    except Exception:
        results['cuDF (NVIDIA GPU)'] = 0

    return results


def start_lab(num_rows):
    rows = int(num_rows)
    path = generate_data(rows)
    perf_results = run_benchmark(path)

    #  الرسم البياني
    fig = go.Figure(data=[
        go.Bar(name='Processing Time',
               x=list(perf_results.keys()),
               y=list(perf_results.values()),
               marker_color=['#E74C3C', '#F1C40F', '#2ECC71'])
    ])

    fig.update_layout(
        title=f"Performance Comparison for {rows:,} Rows",
        template="plotly_dark",
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)'
    )

    if perf_results['cuDF (NVIDIA GPU)'] > 0:
        speedup = perf_results['Pandas (Standard CPU)'] / perf_results['cuDF (NVIDIA GPU)']
        summary = f"✨ Done! At {rows:,} rows, GPU is {speedup:.1f}x faster than Pandas."
    else:
        summary = "⚠️ GPU is not detected. Please enable T4 GPU in Colab settings."

    return fig, summary

# ---  ( Theme) ---

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown(f"""
    # 🚀 Scalable Data Pipeline Dashboard
    ### Developer: **Manar Salman | منار سلمان**

    هذا التطبيق يقارن بين سرعة معالجة البيانات الضخمة باستخدام محركات مختلفة.
    **ملاحظة:** البيانات المستخدمة تم إنشاؤها آلياً (Generated Data) لأغراض الاختبار.
    """)

    with gr.Row():
        with gr.Column(scale=1):
            row_slider = gr.Slider(minimum=500_000, maximum=10_000_000, step=500_000,
                                   label="Select Dataset Size (Rows)", value=1_000_000)
            run_btn = gr.Button("Run Benchmark 🚀", variant="primary")

    with gr.Column():
        plot_out = gr.Plot(label="Visualization Result")
        msg_out = gr.Textbox(label="Benchmark Summary")

    run_btn.click(fn=start_lab, inputs=row_slider, outputs=[plot_out, msg_out])

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://92e0d588b58556eb95.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
